# Suite EVL — Evaluators

`Evaluators.OfAny(term, scope)` computes a term's value with the variables in `scope` bound, and `Evaluators.OfLiteral`,
`OfCall`, `OfVariable`, `OfLet` and `OfQuantifier` evaluate one kind. A term is a value of
`Expressions.Framework.Term`, built in a store, in a transaction (here through `term`, from a writer). Evaluation is
three-valued (absent and incomparable values are unknown, and logic follows Kleene), never coerces native types, and
evaluates only the core operations.

In [ ]:
import { Evaluators as V, Expressions as E, Framework } from "@mbse/expressions";
import * as Basic from "@mbse/expressions/Dialects/Basic";
import { Terms as F } from "@mbse/expressions/Framework";
import { Builders, Errors } from "@mbse/schemas/Framework";
import { assert, equal, raises, same } from "./support.js";

const { KeyError, NotImplementedError, ValueError } = Errors;

const store: any = new Builders.Store();
store.transaction(Framework.install);
store.transaction(Basic.install);

/** The term value `spec` writes, built in a transaction. */
function term(spec: unknown): any {
  return store.transaction(() => store.Expressions.Framework.Term(E.DIALECT.spec(spec)));
}

/** Evaluates the term `spec` writes, with `scope` bound. */
const ev = (spec: unknown, scope: Record<string, unknown> = {}): any => V.OfAny(term(spec), scope);

/** A person, a reference object; a card that reaches its holder by phone or email, a union of value objects. */
function people(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("evl.people/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Person: S.OfObject, Reach: S.OfUnion, Code: S.OfUnion, Card: S.OfObject })
    .spec([
      (_: any) => _.Person.name("evl.Person").ref(true).properties({ name: S.OfStr, age: S.OfInt }),
      (_: any) => _.Reach.name("evl.Reach").branches([
        (t: any) => t.name("phone").schema([S.OfObject((o: any) => o.properties({ kind: S.OfStr, number: S.OfStr }))]),
        (t: any) => t.name("email").schema([S.OfObject((o: any) => o.properties({ kind: S.OfStr, address: S.OfStr }))]),
      ]),
      (_: any) => _.Code.name("evl.Code").branches([ // a union of scalars
        (t: any) => t.name("number").schema([S.OfInt]), (t: any) => t.name("text").schema([S.OfStr]),
        (t: any) => t.name("digits").schema([S.OfContainer((c: any) => c.index([S.OfInt]).element([S.OfInt]))])]),
      (_: any) => _.Card.name("evl.Card").ref(true).properties({ reach: _.Reach, code: _.Code }),
    ])]))]);
}

store.transaction(people);

## EVL-01 · Evaluating literals, variables, lets, `get` and `has`

In [ ]:
const [ann, nobody] = store.transaction(() => [
  store.evl.Person((p: any) => p.name("Ann").age(30n)),
  store.evl.Person((p: any) => p),
]);
let person = E.variable("person");
assert(ev(3n) === 3n && ev(E.literal("x")) === "x" && ev(person, { person: ann }) === ann);
assert(ev(person.age, { person: ann }) === 30n && ev(person.age, { person: nobody }) === null); // absent: unknown
assert(ev(person.has("age"), { person: ann }) === true && ev(person.has("age"), { person: nobody }) === false);
assert(ev(person.age, { person: null }) === null && ev(person.has("age"), { person: null }) === null); // unknown object
assert(ev(E.let_("a", person.age, E.variable("a").add(1n)), { person: ann }) === 31n);
assert(ev(E.let_("a", 1n, E.let_("a", 2n, E.variable("a")))) === 2n); // the innermost binding wins
assert(ev(E.let_("a", 1n, E.variable("a").add(E.let_("a", 2n, E.variable("a"))))) === 3n); // and only in its body
const [one, term_value] = [term(E.literal(1n)), E.variable("term_value")];
assert(ev(term_value.has("literal"), { term_value: one }) === true && ev(term_value.literal.value, { term_value: one }) === 1n); // terms are values too
raises(KeyError, () => ev(E.variable("x")), "variable 'x' is not bound");
raises(TypeError, () => ev(E.operation("get", 1n, "x")), "get expects an object, got int");
raises(TypeError, () => ev(E.operation("has", person, 1n), { person: ann }), "has expects a property name, got int");

## EVL-02 · Comparisons are three-valued and never coerce

In [ ]:
const unit = E.literal(1n);
assert(same([ev(unit.lt(2n)), ev(unit.le(1n)), ev(unit.gt(2n)), ev(unit.ge(1n)), ev(unit.eq(1n)), ev(unit.ne(1n))], [
  true, true, false, true, true, false]));
assert(ev(E.literal("b").gt("a")) === true && ev(E.literal(new Uint8Array([1])).lt(new Uint8Array([2]))) === true);
assert(ev(unit.eq(1.0)) === null && ev(unit.lt(1.5)) === null && ev(unit.ne("1")) === null); // different types
assert(ev(E.literal(true).eq(false)) === false && ev(E.literal(true).lt(false)) === null); // bools are not ordered
assert(ev(E.literal(true).le(true)) === true);
const nan = E.literal(NaN);
assert(ev(nan.eq(NaN)) === true && ev(nan.eq(1.0)) === false && ev(nan.lt(1.0)) === null);
assert(ev(E.literal(-0).lt(0)) === true && ev(E.literal(-0).eq(0)) === false);
const age = E.variable("x").age;
assert(ev(age.eq(1n), { x: nobody }) === null && ev(age.ne(1n), { x: nobody }) === null && ev(age.lt(1n), { x: nobody }) === null);
const [first, second] = [E.variable("a"), E.variable("b")];
assert(ev(first.eq(second), { a: ann, b: ann }) === true && ev(first.eq(second), { a: ann, b: nobody }) === false); // objects: by identity
assert(ev(first.lt(second), { a: ann, b: nobody }) === null && ev(first.eq(1n), { a: ann }) === null);

## EVL-03 · Logic follows Kleene; arithmetic keeps one type

In [ ]:
const [T, F_, U] = [E.literal(true), E.literal(false), E.variable("u")]; // u is unknown

function table(method: string): unknown[][] {
  return [T, F_, U].map((x) => [T, F_, U].map((y) => ev((x as any)[method](y), { u: null })));
}

assert(equal(table("and_"), [[true, false, null], [false, false, false], [null, false, null]]));
assert(equal(table("or_"), [[true, true, true], [true, false, null], [true, null, null]]));
assert(equal(table("implies"), [[true, false, null], [true, true, true], [true, null, null]]));
assert(same([T, F_, U].map((x) => ev(x.not_(), { u: null })), [false, true, null]));
const missing = E.variable("missing"); // never evaluated when the first operand decides
assert(ev(F_.and_(missing)) === false && ev(T.or_(missing)) === true && ev(F_.implies(missing)) === true);
raises(TypeError, () => ev(T.and_(1n)), "and expects bool operands, got int");
raises(TypeError, () => ev(E.literal("x").not_()), "not expects bool operands, got str");
const pair = E.literal(2n);
assert(same([ev(pair.add(3n)), ev(pair.sub(3n)), ev(pair.mul(3n)), ev(pair.neg())], [5n, -1n, 6n, -2n]));
assert(ev(E.literal(1.5).add(1.0)) === 2.5 && ev(E.literal(2n ** 70n).mul(2n)) === 2n ** 71n);
assert(ev(U.add(1n), { u: null }) === null && ev(U.neg(), { u: null }) === null);
raises(TypeError, () => ev(E.literal(1n).add(1.5)), "add expects numbers of one domain, got int and float");
raises(TypeError, () => ev(T.sub(T)), "sub expects numbers of one domain, got bool and bool");
raises(TypeError, () => ev(E.literal("x").neg()), "neg expects a number, got str");

## EVL-04 · Checking: `validate()` finds what evaluation would raise

In [ ]:
assert(same(E.DIALECT.validate(term(E.operation("lt", 1n))), ["lt takes 2 arguments, got 1"]));
const foo = term(E.operation("foo", 1n));
assert(same(E.DIALECT.validate(foo), ["'foo' is not an operation Basic defines"]) && same(E.DIALECT.validate(foo, [], true), ["'foo' is not an operation Basic defines"]));
person = E.variable("person");
const checked = term(E.let_("a", person.age, E.variable("a").ge(18n).and_(person.has("email"))));
assert(E.DIALECT.validate(checked, ["person"], true).length === 0);
assert(same(E.DIALECT.validate(checked), ["value: argument 0: variable 'person' is not bound",
  "body: argument 1: argument 0: variable 'person' is not bound"]));
raises(NotImplementedError, () => V.OfAny(foo), "'foo' is not an operation Basic defines");
raises(TypeError, () => ev(E.operation("lt", 1n)), "lt takes 2 arguments, got 1");
raises(ValueError, () => ev(E.literal(null)), "a literal needs a value");
raises(ValueError, () => ev(E.let_("x", null, null)), "a let needs a value");
raises(ValueError, () => ev(E.let_("x", 1n, null)), "a let needs a body");
// A term is a nested value: written into another, it is copied, so a term never holds itself.
const inner = term(E.operation("not", true));
const outer = term(E.operation("not", inner));
assert(ev(outer) === true && [...F.walk(outer)].length === 3 && [...F.walk(inner)].length === 2);
raises(TypeError, () => E.operation("not", [7n]), "expected a term, a writer or a native value, got [7]");

## EVL-05 · The union of terms names each branch by its kind's tag

In [ ]:
const Term: any = store.registered("Expressions.Framework.Term");
assert(same(Term.branches.map((branch: any) => branch.name), ["literal", "call", "variable", "let", "list", "mapping", "quantifier"]));
assert(same(Term.branches.map((branch: any) => branch.schema.name), [
  "Expressions.Framework.Literal", "Expressions.Framework.Call", "Expressions.Framework.Variable",
  "Expressions.Framework.Let", "Expressions.Framework.List", "Expressions.Framework.Mapping",
  "Expressions.Basic.Quantifier"]));
const samples = [E.literal(1n), E.operation("f", 1n, "x"), E.variable("x"), E.let_("a", 1n, 2n), E.list_(1n), E.mapping(["k", 1n]),
  E.quantifier("all", "x", E.variable("xs"), true)];
assert(same(samples.map((sample) => F.kindOf(term(sample))[0]), Term.branches.map((branch: any) => branch.name)));
assert(Term.validate().length === 0);

## EVL-06 · Each kind has its own entry point, which takes a term of that kind

In [ ]:
const x = E.variable("x");
assert(V.OfLiteral(term(3n)) === 3n && V.OfLiteral(term(E.literal("s"))) === "s");
assert(V.OfVariable(term(x), { x: 2n }) === 2n);
assert(V.OfCall(term(x.add(1n)), { x: 1n }) === 2n && V.OfLet(term(E.let_("y", 1n, E.variable("y").neg()))) === -1n);
assert(V.OfQuantifier(term(x.all("i", E.variable("i").gt(0n))), { x: [1n, 2n] }) === true);
assert(V.OfAny(term(x), { x: null }) === null);
const expression = store.transaction(() => store.Expressions.Framework.Expression((e: any) => e.root(x.add(1n))));
assert(V.OfAny(expression, { x: 1n }) === 2n && V.OfCall(expression.root, { x: 2n }) === 3n); // an expression, its root
raises(TypeError, () => V.OfLet(term(3n)), "expected a let, got a literal");
raises(TypeError, () => V.OfCall(term(x)), "expected a call, got a variable"); // a variable, not a call
raises(TypeError, () => V.OfAny(3n), "not a term: 3");

## EVL-07 · A constraint about a value, bound to a variable, reads value objects and union values

In [ ]:
const [card, mail] = [E.variable("card"), E.variable("mail")]; // a card, and an email, a value object
const [good, phone, sameCard, home, workPhone] = store.transaction(() => [
  store.evl.Card((c: any) => c.reach((r: any) => r.email((e: any) => e.kind("work").address("a@x")))),
  store.evl.Card((c: any) => c.reach((r: any) => r.phone((e: any) => e.number("1")))),
  store.evl.Card((c: any) => c.reach((r: any) => r.email((e: any) => e.kind("work").address("a@x")))),
  store.evl.Card((c: any) => c.reach((r: any) => r.email((e: any) => e.kind("home").address("a@x")))),
  store.evl.Card((c: any) => c.reach((r: any) => r.phone((e: any) => e.kind("work")))),
]);
const email = card.get("reach").get("email"); // a union value is a record of its one branch
assert(ev(card.get("reach").has("email"), { card: good }) === true && ev(card.get("reach").has("phone"), { card: good }) === false);
assert(ev(email.get("kind").eq("work"), { card: good }) === true);
assert(ev(email.get("kind").eq("home"), { card: phone }) === null);
assert(ev(mail.get("kind").eq("phone"), { mail: good.reach.email }) === false && ev(mail.has("kind"), { mail: null }) === null);
assert(ev(mail.get("kind"), { mail: good.reach.email }) === "work"); // a value object's property
assert(ev(mail.eq(mail), { mail: good.reach.email }) === true); // a value object compares deeply, by its properties
const other = E.variable("other");
const reach = card.get("reach").eq(other.get("reach")); // union values: the same branch, equal
assert(ev(reach, { card: good, other: sameCard }) === true);
assert(ev(reach, { card: good, other: home }) === false);
assert(ev(reach, { card: good, other: workPhone }) === null); // another branch
assert(ev(card.eq(other), { card: good, other: good.reach.email }) === null && ev(card.eq(other), { card: good, other: good }) === true);
const coded = store.transaction(() => store.evl.Card((c: any) => c.code((r: any) => r.number(7n))));
assert(ev(card.get("code").get("number"), { card: coded }) === 7n && ev(card.get("code").has("text"), { card: coded }) === false);
const listed = store.transaction(() => store.evl.Card((c: any) => c.code((r: any) => r.digits((k: any) => {
  k.at(0n).as_scalar(1n); // a container branch is written through its visitor
  k.at(1n).as_scalar(2n);
  return k;
}))));
assert(ev(card.get("code").get("digits").count(), { card: listed }) === 2n); // a union's container branch reads as a collection